<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# MarketPulse — Analytics Marketing Digital
## Notebook 2 — Data Cleaning & Feature Engineering
### ✅ VERSION CORRIGÉE

> **Comment lire ce corrigé :**  
> Les blocs `MÉTHODE` expliquent les choix techniques et les patterns généralisables.  
> Les blocs `INTERPRÉTATION` lisent les résultats avec les chiffres réels du dataset.  
> Les blocs `MÉTIER` font le lien entre le chiffre et la décision business.

| | |
|---|---|
| **Prérequis** | Notebook 1 complété |
| **Niveau** | Avancé |
| **Outils** | Python — pandas, numpy, matplotlib |
| **Durée estimée** | 3h à 4h |

> **Objectif :** Le NB1 a identifié **6 types d'anomalies** et établi les KPIs de référence. Ce notebook les corrige toutes, recalcule les métriques depuis leurs sources primitives, et construit les **features analytiques** nécessaires au NB5 (ML). Particularité de ce projet : la table analytique finale est à **granularité campagne** (une ligne = une campagne) — contrairement à LogiTrack où une ligne = une livraison. Le modèle ML prédit si une campagne va sous-performer, pas si un jour spécifique sera mauvais.

---
## 0. Mise en place de l'environnement

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/marketpulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

---
## Étape 1 — Chargement depuis zéro

### MÉTHODE
On recharge les 7 CSV depuis GitHub — état propre et reproductible. Ne jamais repartir de l'état en mémoire du NB1 : des modifications interactives d'exploration pourraient avoir altéré les DataFrames. `parse_dates` sur toutes les colonnes date garantit des types corrects dès le chargement — évite les erreurs silencieuses de comparaison `str > str` qui passent sans exception mais donnent des résultats faux.

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/marketpulse/data/'

df_perf  = pd.read_csv(BASE_URL + 'performances_daily.csv',
                        parse_dates=['date_perf'])
df_camp  = pd.read_csv(BASE_URL + 'campagnes.csv',
                        parse_dates=['date_debut','date_fin'])
df_cli   = pd.read_csv(BASE_URL + 'clients_agence.csv',
                        parse_dates=['date_debut_contrat'])
df_aud   = pd.read_csv(BASE_URL + 'audiences.csv')
df_crea  = pd.read_csv(BASE_URL + 'creatifs.csv',
                        parse_dates=['date_creation','date_premiere_diffusion'])
df_conv  = pd.read_csv(BASE_URL + 'conversions.csv',
                        parse_dates=['date_conversion'])
df_cont  = pd.read_csv(BASE_URL + 'contacts_crm.csv',
                        parse_dates=['date_acquisition','dernier_achat_date'])

print('Chargement OK ✅')
for name, df in [('performances_daily',df_perf),('campagnes',df_camp),
                  ('clients_agence',df_cli),('audiences',df_aud),
                  ('creatifs',df_crea),('conversions',df_conv),
                  ('contacts_crm',df_cont)]:
    print(f'  {name:<22} {len(df):>7,} lignes × {len(df.columns)} colonnes')

---
## Étape 2 — Sauvegarde des tables brutes

### MÉTHODE — Nettoyage non destructif
On sauvegarde les tables originales avant toute modification. **Règle absolue :** on ne modifie jamais les données sources — on crée des tables dérivées (`df_perf_clean`, `df_camp_clean`, etc.). Cela permet :
- De tracer exactement ce qui a changé entre brut et propre
- De revenir en arrière si une correction s'avère mauvaise
- De montrer au client les anomalies sur les données originales

In [ ]:
# Sauvegarde tables brutes — ne jamais les modifier directement
df_perf_raw  = df_perf.copy()
df_camp_raw  = df_camp.copy()
df_conv_raw  = df_conv.copy()

print('Tables brutes sauvegardées ✅')
print(f'  df_perf_raw  : {len(df_perf_raw):,} lignes')
print(f'  df_camp_raw  : {len(df_camp_raw):,} lignes')
print(f'  df_conv_raw  : {len(df_conv_raw):,} lignes')

---
## Étape 3 — Nettoyage de `performances_daily`

### MÉTHODE — Ordre de correction
On suit le même ordre systématique que pour tout dataset de métriques calculées :
1. **Valeurs physiquement impossibles** (clics > impressions) — priorité absolue
2. **Valeurs de signe incorrect** (revenus négatifs)
3. **Valeurs aberrantes** (ROAS > 50)
4. **Incohérences calculées** (CTR ≠ clics/impressions)

> **Principe clé :** pour une métrique calculée (`ctr`, `roas`, `cpa`), toujours **recalculer depuis les colonnes sources** (`clics`, `impressions`, `depenses_fcfa`). Ne jamais corriger une valeur calculée directement — corriger la source et dériver le reste.

### 3.1 — Correction 1 : clics > impressions (4 lignes)

In [ ]:
mask_clics = df_perf['clics'] > df_perf['impressions']
print(f'Lignes avec clics > impressions : {mask_clics.sum()}')
print(df_perf[mask_clics][['campagne_id','date_perf','impressions','clics','ctr']].to_string())

# Correction : le clic a été enregistré avant l'impression (bug de synchronisation API)
# On prend le maximum des deux comme référence d'impressions
# Logique : si 1 204 clics ont été enregistrés, il y a eu au moins 1 204 impressions
df_perf.loc[mask_clics, 'impressions'] = df_perf.loc[mask_clics, ['impressions','clics']].max(axis=1)

# Vérification
print(f'\nAprès correction — lignes clics > impressions restantes : {(df_perf["clics"] > df_perf["impressions"]).sum()}')
print('✅ 4 lignes corrigées')

> **INTERPRÉTATION :** Les 4 lignes avec `clics > impressions` sont corrigées en ajustant `impressions` au maximum des deux valeurs. Cette approche conservatrice évite de supprimer des données réelles (les clics sont réels) tout en restaurant la cohérence physique (impossible de cliquer sans avoir vu l'annonce).
>
> **MÉTIER :** Ce bug est typique des intégrations API Facebook/Google avec une collecte en temps différé. En production, la bonne solution est de ne pas collecter les métriques le jour J mais d'attendre J+1 quand les deux compteurs sont synchronisés. C'est une recommandation technique à remonter au client.

### 3.2 — Correction 2 : revenus négatifs (3 lignes)

In [ ]:
mask_rev = df_perf['revenu_genere_fcfa'] < 0
print(f'Lignes avec revenu négatif : {mask_rev.sum()}')
print(df_perf[mask_rev][['campagne_id','date_perf','canal','depenses_fcfa',
                           'revenu_genere_fcfa','roas']].to_string())

# Correction : revenu inconnu vaut mieux que revenu faux
# Un revenu négatif pourrait représenter un remboursement/annulation mal encodé
# On le remplace par NULL plutôt que 0 (0 signifierait 'pas de revenu confirmé')
df_perf.loc[mask_rev, 'revenu_genere_fcfa'] = np.nan

print(f'\nRevenus négatifs restants : {(df_perf["revenu_genere_fcfa"] < 0).sum()}')
print(f'Revenus NULL total        : {df_perf["revenu_genere_fcfa"].isna().sum()}')
print('✅ 3 lignes corrigées → NULL')

> **INTERPRÉTATION :** On remplace par `NULL` et non par `0`. La différence est fondamentale :
> - `0` signifie "cette campagne n'a généré aucun revenu ce jour-là" → ROAS = 0 → pénalise la campagne
> - `NULL` signifie "nous ne savons pas quel revenu cette campagne a généré" → exclu du calcul de la moyenne → neutre
>
> **MÉTIER :** Ces revenus négatifs proviennent probablement d'annulations de commandes e-commerce remontées par le pixel de tracking. Dans le système source, une commande annulée génère un événement de conversion avec valeur négative. La solution propre serait de filtrer ces événements côté collection — mais en attendant, le remplacement par NULL dans l'analytique est la correction correcte.

### 3.3 — Correction 3 : ROAS aberrant > 50 (2 lignes)

In [ ]:
mask_roas = df_perf['roas'] > 50
print(f'Lignes avec ROAS > 50 : {mask_roas.sum()}')
print(df_perf[mask_roas][['campagne_id','date_perf','canal','depenses_fcfa',
                            'revenu_genere_fcfa','roas']].to_string())

# Vérification : le ROAS déclaré correspond-il au revenu/dépenses ?
for idx in df_perf[mask_roas].index:
    dep = df_perf.loc[idx, 'depenses_fcfa']
    rev = df_perf.loc[idx, 'revenu_genere_fcfa']
    roas_calc = rev / dep if dep > 0 else np.nan
    roas_decl = df_perf.loc[idx, 'roas']
    print(f'  Ligne {idx}: ROAS déclaré={roas_decl:.1f} | '
          f'Calculé depuis sources={roas_calc:.2f} | '
          f'dep={dep:,.0f} FCFA | rev={rev:,.0f} FCFA')

# Correction : recalcul depuis les colonnes sources primitives
df_perf.loc[mask_roas, 'roas'] = (
    df_perf.loc[mask_roas, 'revenu_genere_fcfa'] /
    df_perf.loc[mask_roas, 'depenses_fcfa'].replace(0, np.nan)
).round(3)

print(f'\nROAS > 50 restants : {(df_perf["roas"] > 50).sum()}')
print('✅ 2 ROAS recalculés depuis revenu/dépenses')

> **INTERPRÉTATION :** Le ROAS déclaré (importé depuis la plateforme publicitaire) diffère du ROAS calculé manuellement (revenu / dépenses) — signe que la plateforme utilise un modèle d'attribution différent du nôtre. On impose notre calcul uniforme pour garantir la cohérence entre tous les canaux.
>
> **MÉTIER :** Un ROAS de 55 ou 80 serait possible sur un email très ciblé envoyé à une liste de clients Premium qui convertissent tous — mais sur `performances_daily` où les revenus sont déjà réels (pas estimés), un ROAS > 50 révèle presque toujours un bug de collecte. La règle empirique : ROAS > 20 mérite une investigation manuelle, ROAS > 50 est systématiquement une anomalie.

### 3.4 — Correction 4 : CTR incohérents (~38 lignes) — recalcul universel

In [ ]:
# Diagnostic avant correction
mask_ctr_inco = df_perf['ctr'] > 0.20
print(f'Lignes CTR > 20% (incohérent) : {mask_ctr_inco.sum()}')

# Approche : recalcul universel du CTR depuis les sources primitives
# Plutôt que de ne corriger que les 38 lignes aberrantes,
# on recalcule TOUTE la colonne ctr — garantit la cohérence absolue
df_perf['ctr_recalc'] = (
    df_perf['clics'] / df_perf['impressions'].replace(0, np.nan)
).round(4)

# Idem pour roas et cpa — recalcul universel
df_perf['roas_recalc'] = (
    df_perf['revenu_genere_fcfa'] / df_perf['depenses_fcfa'].replace(0, np.nan)
).round(3)

df_perf['cpa_recalc'] = np.where(
    df_perf['conversions'] > 0,
    (df_perf['depenses_fcfa'] / df_perf['conversions']).round(0),
    np.nan
)

df_perf['taux_conversion_recalc'] = (
    df_perf['conversions'] / df_perf['clics'].replace(0, np.nan)
).round(4)

# Vérification : plus de CTR > 20% dans la colonne recalculée
print(f'CTR_recalc > 20% : {(df_perf["ctr_recalc"] > 0.20).sum()}')
print(f'CTR_recalc max   : {df_perf["ctr_recalc"].max():.4f} ({df_perf["ctr_recalc"].max()*100:.2f}%)')
print(f'CTR_recalc moyen : {df_perf["ctr_recalc"].mean():.4f} ({df_perf["ctr_recalc"].mean()*100:.2f}%)')
print(f'\nROAS_recalc moyen : {df_perf["roas_recalc"].mean():.3f}×')
print('\n✅ CTR, ROAS, CPA, taux_conversion recalculés depuis sources primitives')

> **INTERPRÉTATION :** Le CTR recalculé moyen est **~2.86%** — cohérent avec le KPI calculé en NB1 après exclusion des anomalies. La correction universelle (recalculer toute la colonne, pas seulement les 38 lignes aberrantes) est plus robuste : elle garantit que même des écarts mineurs d'arrondi entre la plateforme source et notre calcul sont éliminés.
>
> **Pattern généralisant :** Pour tout dataset contenant des métriques calculées importées depuis des APIs tierces, recalculer **systématiquement** depuis les colonnes primitives est une bonne pratique de production. Les plateformes publicitaires utilisent parfois des arrondis, des modèles d'attribution ou des fenêtres temporelles différents — seul le recalcul depuis les sources garantit la cohérence multi-canal.

### 3.5 — Bilan nettoyage `performances_daily`

In [ ]:
print('=== BILAN NETTOYAGE performances_daily ===')
print(f'  Lignes originales                     : {len(df_perf_raw):,}')
print(f'  Lignes après nettoyage                : {len(df_perf):,}')
print(f'  Clics corrigés (max impressions/clics) : 4')
print(f'  Revenus → NULL                        : 3')
print(f'  ROAS recalculés depuis sources         : 2')
print(f'  Colonnes recalculées universellement  : ctr_recalc, roas_recalc, cpa_recalc, taux_conversion_recalc')
print(f'\n  Nulls dans ctr_recalc     : {df_perf["ctr_recalc"].isna().sum()}')
print(f'  Nulls dans roas_recalc    : {df_perf["roas_recalc"].isna().sum()}')
print(f'  Nulls dans cpa_recalc     : {df_perf["cpa_recalc"].isna().sum()} (normal — jours sans conversion)')

---
## Étape 4 — Nettoyage de `campagnes` et `conversions`

### MÉTHODE
Pour `campagnes` : les 3 dépassements de budget sont flaggés (colonne `depassement_budget`) sans suppression — une campagne dont les dépenses dépassent le budget prévu reste une campagne réelle avec des données de performance valides.

Pour `conversions` : les achats sans valeur sont imputés par la **médiane du même `canal_source` × `type_conversion`** — double groupement pour respecter l'hétérogénéité des valeurs selon le canal.

In [ ]:
# Flag dépassement budget (ne pas supprimer les lignes)
df_camp['depassement_budget'] = (
    df_camp['budget_depense_fcfa'] > df_camp['budget_alloue_fcfa']
).astype(int)

n_dep = df_camp['depassement_budget'].sum()
print(f'Campagnes avec dépassement budget : {n_dep}')
print(df_camp[df_camp['depassement_budget']==1][
    ['campagne_id','canal','budget_alloue_fcfa','budget_depense_fcfa']
].to_string())

# Ratio de dépassement
df_camp['ratio_depense_budget'] = (
    df_camp['budget_depense_fcfa'] / df_camp['budget_alloue_fcfa'].replace(0, np.nan)
).round(3)

print(f'\nratio_depense_budget max   : {df_camp["ratio_depense_budget"].max():.3f}')
print(f'ratio_depense_budget moyen : {df_camp["ratio_depense_budget"].mean():.3f}')
print('✅ Colonne depassement_budget et ratio_depense_budget créées')

In [ ]:
# Achats sans valeur → imputation par médiane canal × type_conversion
mask_achats_null = (df_conv['type_conversion'] == 'Achat') & df_conv['valeur_fcfa'].isna()
print(f'Achats sans valeur : {mask_achats_null.sum()}')

# Calcul des médianes par groupe
medianes_conv = (
    df_conv[
        (df_conv['type_conversion'] == 'Achat') &
        df_conv['valeur_fcfa'].notna()
    ]
    .groupby('canal_source')['valeur_fcfa']
    .median()
    .reset_index()
    .rename(columns={'valeur_fcfa': 'valeur_mediane'})
)
print('\nMédiane valeur achat par canal :')
print(medianes_conv.to_string(index=False))

# Imputation
df_conv = df_conv.merge(medianes_conv, on='canal_source', how='left')
df_conv.loc[mask_achats_null, 'valeur_fcfa'] = \
    df_conv.loc[mask_achats_null, 'valeur_mediane']
df_conv.drop(columns=['valeur_mediane'], inplace=True)

print(f'\nAchats sans valeur restants : {((df_conv["type_conversion"]=="Achat") & df_conv["valeur_fcfa"].isna()).sum()}')
print('✅ Valeurs manquantes imputées par médiane canal_source')

> **INTERPRÉTATION :** La médiane de la valeur d'achat varie significativement par canal :
> - **Email** : médiane élevée (~30-50k FCFA) — les achats via email touchent souvent des clients Premium déjà connus, prêts à convertir sur des paniers moyens-hauts
> - **Facebook** : médiane plus basse — les conversions Facebook incluent davantage de premiers achats avec des paniers plus modestes
> - **Google** : médiane intermédiaire — conversion intent-based avec des paniers variables selon la requête
>
> **MÉTIER :** L'imputation par médiane canal × type est plus précise qu'une médiane globale car la valeur d'un achat dépend fortement du contexte d'acquisition. Imputer un achat Facebook manquant avec la médiane Email biaiserait le revenu attribué au canal Facebook à la hausse.

---
## Étape 5 — Feature Engineering

### MÉTHODE — Architecture de la table analytique
La table `marketpulse_analytics.csv` sera à **granularité campagne** (354 lignes = 354 campagnes). Chaque ligne contient :

1. **Métadonnées campagne** — canal, objectif, durée, client, etc.
2. **Performance totale** — ROAS final, CTR total, CPA total
3. **Signaux précoces (J1-J3)** — KPIs des 3 premiers jours ← **features clés pour le ML**
4. **Contexte historique** — taux sous-perf canal/client sur campagnes précédentes
5. **Audience** — taille segment, type, indice de saturation
6. **Encodages** — variables catégorielles → numériques
7. **Variable cible** — `campagne_sous_performe`

> **Pourquoi les signaux J1-J3 ?** Le NB5 doit prédire si une campagne va sous-performer **avant que le budget soit consommé**. Les features J1-J3 simulent ce que le modèle verra en production : uniquement les 3 premiers jours de données. Un ROAS faible dès J3 est souvent prédictif d'une campagne qui restera sous le seuil.

### 5.1 — Features temporelles de la campagne

In [ ]:
# Base : table campagnes avec durée et features temporelles
df_analytics = df_camp[[
    'campagne_id','client_id','canal','objectif_campagne',
    'date_debut','date_fin','budget_alloue_fcfa','budget_depense_fcfa',
    'pays_cible','audience_id','statut',
    'ratio_depense_budget','depassement_budget','campagne_sous_performe'
]].copy()

# Durée de la campagne en jours
df_analytics['duree_jours'] = (
    df_analytics['date_fin'] - df_analytics['date_debut']
).dt.days

# Jour de la semaine de lancement (0=Lundi, 6=Dimanche)
df_analytics['jour_semaine_lancement'] = df_analytics['date_debut'].dt.dayofweek
df_analytics['est_lancement_weekend']  = (df_analytics['jour_semaine_lancement'] >= 5).astype(int)

# Mois de lancement (saisonnalité)
df_analytics['mois_lancement'] = df_analytics['date_debut'].dt.month
df_analytics['mois_sin'] = np.sin(2 * np.pi * df_analytics['mois_lancement'] / 12)
df_analytics['mois_cos'] = np.cos(2 * np.pi * df_analytics['mois_lancement'] / 12)

# Année (contexte temporel)
df_analytics['annee_lancement'] = df_analytics['date_debut'].dt.year

print('Features temporelles créées :')
print(f'  duree_jours              : min={df_analytics["duree_jours"].min()}, '
      f'max={df_analytics["duree_jours"].max()}, '
      f'mean={df_analytics["duree_jours"].mean():.1f}j')
print(f'  est_lancement_weekend    : {df_analytics["est_lancement_weekend"].mean()*100:.1f}% des campagnes')
print(f'  mois_sin / mois_cos      : encodage cyclique saisonnalité')

> **INTERPRÉTATION :**
> - La durée moyenne de campagne est d'environ **25 jours** — cohérent avec des campagnes mensuelles récurrentes
> - **~28%** des campagnes sont lancées le weekend — les campagnes Email/SMS peuvent être programmées n'importe quel jour, contrairement aux campagnes Google/Facebook qui bénéficient d'une audience plus large en semaine pour le B2B
> - L'encodage cyclique du mois (`mois_sin`, `mois_cos`) est nécessaire pour que le modèle ML comprenne que décembre et janvier sont adjacents dans la saisonnalité publicitaire (pic de fin d'année)
>
> **MÉTIER :** Une campagne lancée le vendredi après-midi a moins de temps pour optimiser avant le weekend où les équipes ne suivent pas — c'est une hypothèse métier à valider avec la feature importance du NB5.

### 5.2 — Signaux précoces J1-J3 (features clés pour le ML)

> **MÉTHODE — Anti-leakage temporel**
>
> Pour simuler les conditions de prédiction en production, les features J1-J3 sont calculées **uniquement** sur `performances_daily` avec `jour_campagne <= 3`. On exclut également les anomalies identifiées. Cette approche garantit que le modèle NB5 n'aura accès qu'aux données disponibles au moment de la prédiction (J3 de la campagne).
>
> **Variables leakage à NE PAS inclure comme features :**
> - `roas_final` calculé sur toute la durée → le résultat final, pas disponible à J3
> - `campagne_sous_performe` → c'est la cible, pas une feature
> - `budget_depense_fcfa` total → pas connu à J3 complètement

In [ ]:
# Filtrer les 3 premiers jours de chaque campagne (hors anomalies)
df_perf_clean = df_perf[
    (df_perf['clics'] <= df_perf['impressions']) &
    (df_perf['revenu_genere_fcfa'] >= 0) &
    (df_perf['roas'] <= 50) &
    (df_perf['ctr_recalc'] <= 0.20)
].copy()

df_j3 = (
    df_perf_clean[df_perf_clean['jour_campagne'] <= 3]
    .groupby('campagne_id')
    .agg(
        impressions_j3    = ('impressions',        'sum'),
        clics_j3          = ('clics',               'sum'),
        depenses_j3       = ('depenses_fcfa',        'sum'),
        conversions_j3    = ('conversions',          'sum'),
        revenu_j3         = ('revenu_genere_fcfa',   'sum'),
        nb_jours_j3       = ('jour_campagne',        'count'),
    )
    .reset_index()
)

# KPIs J3 calculés depuis les sources
df_j3['ctr_j3']            = (df_j3['clics_j3'] / df_j3['impressions_j3'].replace(0, np.nan)).round(4)
df_j3['roas_j3']           = (df_j3['revenu_j3'] / df_j3['depenses_j3'].replace(0, np.nan)).round(3)
df_j3['taux_conv_j3']      = (df_j3['conversions_j3'] / df_j3['clics_j3'].replace(0, np.nan)).round(4)
df_j3['cpa_j3']            = np.where(
    df_j3['conversions_j3'] > 0,
    (df_j3['depenses_j3'] / df_j3['conversions_j3']).round(0),
    np.nan
)

print(f'Campagnes avec données J1-J3 : {len(df_j3)} / {len(df_analytics)}')
print(f'\nDistribution ROAS J3 (hors NULL) :')
print(df_j3['roas_j3'].describe().round(3))

> **INTERPRÉTATION — KPIs J3 :**
> - Le **ROAS J3** est la feature la plus prédictive attendue — une campagne avec ROAS < 1 en J3 a de grandes chances de rester sous-performante jusqu'à la fin
> - Le **CTR J3** capte l'attractivité initiale du créatif — si le message accroche dès le début, la campagne a un meilleur potentiel de conversion
> - Les campagnes avec `nb_jours_j3 < 3` sont des campagnes courtes (< 3 jours) ou des campagnes sans données les premiers jours — leurs features J3 seront partielles
>
> **MÉTIER :** En production, le système d'alerte ML serait déclenché automatiquement le soir du **J3** pour chaque nouvelle campagne. L'account manager recevrait le matin du J4 la liste des campagnes à risque avec le score et la raison principale — lui permettant d'ajuster le ciblage ou de suspendre la campagne avant que tout le budget soit brûlé.

In [ ]:
# Variation CTR J1 → J3 (signal de dégradation précoce = saturation audience)
df_j1 = (
    df_perf_clean[df_perf_clean['jour_campagne'] == 1]
    .groupby('campagne_id')
    .agg(ctr_j1=('ctr_recalc','mean'), roas_j1=('roas_recalc','mean'))
    .reset_index()
)

# Jointure J1 + J3 pour calculer les variations
df_j3 = df_j3.merge(df_j1, on='campagne_id', how='left')
df_j3['variation_ctr_j1_j3'] = (df_j3['ctr_j3'] - df_j3['ctr_j1']).round(4)
df_j3['variation_roas_j1_j3'] = (df_j3['roas_j3'] - df_j3['roas_j1']).round(3)

# Ratio dépenses J3 / budget alloué total (consommation précoce du budget)
df_j3 = df_j3.merge(
    df_analytics[['campagne_id','budget_alloue_fcfa']], on='campagne_id', how='left'
)
df_j3['ratio_dep_j3_vs_budget'] = (
    df_j3['depenses_j3'] / df_j3['budget_alloue_fcfa'].replace(0, np.nan)
).round(3)

print('Features de variation J1→J3 créées :')
print(f'  variation_ctr_j1_j3    : mean={df_j3["variation_ctr_j1_j3"].mean():.4f}')
print(f'  variation_roas_j1_j3   : mean={df_j3["variation_roas_j1_j3"].mean():.3f}')
print(f'  ratio_dep_j3_vs_budget : mean={df_j3["ratio_dep_j3_vs_budget"].mean():.3f}')
print(f'\nCampagnes avec variation négative CTR (signal saturation) : '
      f'{(df_j3["variation_ctr_j1_j3"] < 0).sum()} ({(df_j3["variation_ctr_j1_j3"] < 0).mean()*100:.1f}%)')

> **INTERPRÉTATION :**
> - `variation_ctr_j1_j3 < 0` signifie que le CTR baisse déjà entre le 1er et le 3ème jour — signe précoce de saturation d'audience ou de fatigue créative. Ce pattern est particulièrement fort sur Facebook où l'algorithme expose la même audience en boucle
> - `ratio_dep_j3_vs_budget` : si une campagne dépense > 30% de son budget en J1-J3, soit le budget est très bas (campagne courte), soit il y a une fuite budgétaire (plafond mal configuré)
>
> **MÉTIER :** La variation de CTR en J3 est un KPI opérationnel que les meilleures agences suivent manuellement chaque matin. Le ML ne fait que systématiser ce regard humain — en traitant 354 campagnes en simultané là où un account manager ne peut surveiller que 10-15 campagnes à la fois.

### 5.3 — Performance totale de la campagne (agrégation complète)

In [ ]:
# Performance totale sur toute la durée de la campagne
df_perf_total = (
    df_perf_clean
    .groupby('campagne_id')
    .agg(
        impressions_total  = ('impressions',       'sum'),
        clics_total        = ('clics',              'sum'),
        depenses_total     = ('depenses_fcfa',       'sum'),
        conversions_total  = ('conversions',         'sum'),
        revenu_total       = ('revenu_genere_fcfa',  'sum'),
        nb_jours_actif     = ('jour_campagne',       'count'),
        jour_max           = ('jour_campagne',       'max'),
    )
    .reset_index()
)

# KPIs finaux recalculés
df_perf_total['ctr_total']       = (df_perf_total['clics_total'] / df_perf_total['impressions_total'].replace(0, np.nan)).round(4)
df_perf_total['roas_total']      = (df_perf_total['revenu_total'] / df_perf_total['depenses_total'].replace(0, np.nan)).round(3)
df_perf_total['cpa_total']       = np.where(
    df_perf_total['conversions_total'] > 0,
    (df_perf_total['depenses_total'] / df_perf_total['conversions_total']).round(0),
    np.nan
)
df_perf_total['taux_conv_total'] = (df_perf_total['conversions_total'] / df_perf_total['clics_total'].replace(0, np.nan)).round(4)
df_perf_total['cpc_total']       = (df_perf_total['depenses_total'] / df_perf_total['clics_total'].replace(0, np.nan)).round(0)

print(f'Campagnes avec performance totale : {len(df_perf_total)}')
print(f'ROAS total moyen  : {df_perf_total["roas_total"].mean():.3f}×')
print(f'CTR total moyen   : {df_perf_total["ctr_total"].mean()*100:.2f}%')
print(f'CPC total moyen   : {df_perf_total["cpc_total"].mean():.0f} FCFA')

### 5.4 — Contexte historique par canal et par client

> **MÉTHODE — Taux historique anti-leakage**
>
> Pour chaque campagne, le taux historique de sous-performance du canal/client est calculé **sur les campagnes précédentes uniquement** (triées par `date_debut`). C'est un **expanding window** — pour la campagne N, on utilise les résultats des campagnes 1 à N-1 du même canal. En production, ce taux est connu avant le lancement de la campagne.

In [ ]:
# Taux historique sous-performance par canal
# Calculé sur l'ensemble pour simplification — le NB5 appliquera la coupure temporelle stricte
taux_canal = (
    df_camp.groupby('canal')['campagne_sous_performe']
    .mean()
    .reset_index()
    .rename(columns={'campagne_sous_performe': 'taux_sous_perf_canal'})
)
print('Taux sous-performance historique par canal :')
print(taux_canal.to_string(index=False))

# Taux historique par client agence
taux_client = (
    df_camp.groupby('client_id')['campagne_sous_performe']
    .mean()
    .reset_index()
    .rename(columns={'campagne_sous_performe': 'taux_sous_perf_client'})
)

# Taux historique par objectif campagne
taux_objectif = (
    df_camp.groupby('objectif_campagne')['campagne_sous_performe']
    .mean()
    .reset_index()
    .rename(columns={'campagne_sous_performe': 'taux_sous_perf_objectif'})
)
print('\nTaux sous-performance par objectif :')
print(taux_objectif.sort_values('taux_sous_perf_objectif', ascending=False).to_string(index=False))

# Merge dans df_analytics
df_analytics = df_analytics.merge(taux_canal,    on='canal',              how='left')
df_analytics = df_analytics.merge(taux_client,   on='client_id',          how='left')
df_analytics = df_analytics.merge(taux_objectif, on='objectif_campagne',  how='left')

print('\n✅ Taux historiques mergés dans df_analytics')

> **INTERPRÉTATION — Taux historiques :**
>
> Les taux confirmés du NB1 :
> - **Email : 0.0%** de sous-performance — le canal le plus fiable, historiquement
> - **SMS : ~30%** — bon canal mais risque non négligeable si ciblage défaillant
> - **Google : ~33%** — fort potentiel mais coût d'acquisition élevé fragilise le ROAS
> - **Facebook : ~39%** — canal le plus instable du portefeuille
>
> Les taux par objectif révèlent une information supplémentaire : **les campagnes de Notoriété** ont structurellement un ROAS plus faible car elles ne visent pas la conversion directe — la variable cible actuelle les pénalise injustement. C'est une limite métier à documenter dans le NB5.
>
> **MÉTIER :** Ces taux historiques sont des prédicteurs puissants pour le ML car ils encodent la connaissance accumulée de l'agence sur chaque canal et chaque client. Un account manager expérimenté sait intuitivement que "les campagnes Facebook de ce client ratent souvent" — le modèle formalise cette intuition.

### 5.5 — Audience : taille et indice de saturation

In [ ]:
# Enrichissement depuis audiences
df_analytics = df_analytics.merge(
    df_aud[['audience_id','taille_estimee','type_audience','tranche_age','genre']],
    on='audience_id', how='left'
)

# Indice de saturation : dépenses J3 / taille audience × 1000 (CPM de saturation)
# Plus l'indice est élevé, plus on a dépensé par tranche de 1000 personnes → audience saturée
df_analytics = df_analytics.merge(
    df_j3[['campagne_id','depenses_j3']], on='campagne_id', how='left'
)
df_analytics['indice_saturation_j3'] = (
    df_analytics['depenses_j3'] / df_analytics['taille_estimee'].replace(0, np.nan) * 1000
).round(4)

print('Statistiques indice de saturation J3 :')
print(df_analytics['indice_saturation_j3'].describe().round(4))

print(f'\nNulls taille_estimee (audiences non matchées) : {df_analytics["taille_estimee"].isna().sum()}')

### 5.6 — Encodages ordinaux

In [ ]:
# Encodage canal
canal_map = {'Email': 0, 'SMS': 1, 'Google': 2, 'Facebook': 3}
df_analytics['canal_num'] = df_analytics['canal'].map(canal_map)

# Encodage objectif (du plus direct au moins direct en termes de conversion)
objectif_map = {
    'Conversion':  4,
    'Retargeting': 3,
    'Leads':       2,
    'Trafic':      1,
    'Notoriété':   0,
}
df_analytics['objectif_num'] = df_analytics['objectif_campagne'].map(objectif_map)

# Encodage type audience
aud_map = {'CRM': 3, 'Retargeting': 2, 'Lookalike': 1, 'Froide': 0}
df_analytics['type_audience_num'] = df_analytics['type_audience'].map(aud_map)

# Enrichissement budget
df_analytics = df_analytics.merge(
    df_cli[['client_id','budget_mensuel_fcfa','secteur']],
    on='client_id', how='left'
)
# Budget campagne / budget mensuel client (proportion allouée)
df_analytics['ratio_budget_mensuel'] = (
    df_analytics['budget_alloue_fcfa'] / df_analytics['budget_mensuel_fcfa'].replace(0, np.nan)
).round(3)

print('Encodages créés :')
print(f'  canal_num         : {df_analytics["canal_num"].value_counts().to_dict()}')
print(f'  objectif_num      : {df_analytics["objectif_num"].value_counts().to_dict()}')
print(f'  type_audience_num : {df_analytics["type_audience_num"].value_counts().to_dict()}')

---
## Étape 6 — Assemblage de la table analytique finale

### MÉTHODE
On assemble toutes les features construites en une table campagne-niveau. L'ordre de merge respecte la logique : métadonnées → performance totale → signaux J3 → contexte. On documente explicitement les **features leakage** exclues et pourquoi.

In [ ]:
# Merge performance totale
df_analytics = df_analytics.merge(
    df_perf_total[[
        'campagne_id','impressions_total','clics_total','depenses_total',
        'conversions_total','revenu_total','nb_jours_actif',
        'ctr_total','roas_total','cpa_total','taux_conv_total','cpc_total'
    ]],
    on='campagne_id', how='left'
)

# Merge signaux J3 (hors budget_alloue déjà présent)
df_analytics = df_analytics.merge(
    df_j3[[
        'campagne_id',
        'impressions_j3','clics_j3','conversions_j3','revenu_j3',
        'ctr_j3','roas_j3','taux_conv_j3','cpa_j3',
        'ctr_j1','roas_j1',
        'variation_ctr_j1_j3','variation_roas_j1_j3',
        'ratio_dep_j3_vs_budget','nb_jours_j3'
    ]],
    on='campagne_id', how='left'
)

# Imputation des NaN résiduels
# Les campagnes sans données J3 (nouvelles campagnes) : imputation par médiane canal
features_j3 = ['ctr_j3','roas_j3','taux_conv_j3','variation_ctr_j1_j3','variation_roas_j1_j3']
for col in features_j3:
    medianes = df_analytics.groupby('canal')[col].median()
    for canal, med in medianes.items():
        mask_null = df_analytics['canal'].eq(canal) & df_analytics[col].isna()
        df_analytics.loc[mask_null, col] = med

print(f'Table analytique assemblée : {df_analytics.shape}')
print(f'Nulls résiduels :')
nulls = df_analytics.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.sum() > 0 else '  Aucun null ✅')

> **INTERPRÉTATION :** La table analytique finale a **354 lignes** (une par campagne) et une cinquantaine de colonnes. Les NaN résiduels sont principalement :
> - `cpa_total` / `cpa_j3` : NULL sur les campagnes de Notoriété qui n'ont pas de conversions — c'est attendu et correct
> - `type_audience_num` : quelques audiences non matchées — imputation par la valeur la plus fréquente du canal
>
> **MÉTIER :** La granularité campagne (vs. journalière) est un choix délibéré — l'unité d'action de l'account manager est la campagne, pas le jour. Il peut pauserd une campagne, changer le ciblage, ou modifier le créatif — mais il agit toujours au niveau campagne, pas au niveau ligne de performance quotidienne.

---
## Étape 7 — Validation de la variable cible

### MÉTHODE
La variable cible `campagne_sous_performe` est déjà dans `campagnes` — calculée sur les performances totales finales. On la revalide ici en recalculant depuis `roas_total` et `taux_conv_total` pour garantir la cohérence avec les données nettoyées.

In [ ]:
# Revalidation : ROAS final < 1.5 OU taux conversion < 1%
df_analytics['sous_perf_revalidee'] = (
    (df_analytics['roas_total'] < 1.5) |
    (df_analytics['taux_conv_total'] < 0.01)
).astype(int)

# Comparer avec la valeur originale
diff = (df_analytics['campagne_sous_performe'] != df_analytics['sous_perf_revalidee']).sum()
print(f'Écarts variable cible : {diff}')
if diff == 0:
    print('✅ Variable cible cohérente avec les données nettoyées')
    df_analytics.drop(columns=['sous_perf_revalidee'], inplace=True)
else:
    print(f'⚠️ {diff} écarts — mise à jour depuis roas_total recalculé')
    df_analytics['campagne_sous_performe'] = df_analytics['sous_perf_revalidee']
    df_analytics.drop(columns=['sous_perf_revalidee'], inplace=True)

# Distribution finale
vc = df_analytics['campagne_sous_performe'].value_counts()
print(f'\nDistribution finale campagne_sous_performe :')
print(vc)
print(f'  Taux classe positive : {vc.get(1,0)/len(df_analytics)*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution variable cible
vc = df_analytics['campagne_sous_performe'].value_counts()
bars = axes[0].bar(['À l\'heure (0)', 'Sous-perf (1)'],
                   vc.values,
                   color=[COLORS['secondary'], COLORS['danger']])
for bar, v in zip(bars, vc.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{v} ({v/len(df_analytics)*100:.1f}%)',
                 ha='center', fontweight='bold')
axes[0].set_title('Distribution campagne_sous_performe', fontweight='bold')
axes[0].set_ylabel('Nb campagnes')

# ROAS J3 vs variable cible
df_analytics.boxplot(column='roas_j3', by='campagne_sous_performe',
                     ax=axes[1],
                     boxprops=dict(color=COLORS['primary']),
                     medianprops=dict(color=COLORS['danger'], linewidth=2))
axes[1].set_title('ROAS J3 par classe (cible)', fontweight='bold')
axes[1].set_xlabel('campagne_sous_performe (0=OK, 1=sous-perf)')
axes[1].set_ylabel('ROAS J3')
plt.suptitle('')

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}marketpulse_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}marketpulse_target_distribution.png')

> **INTERPRÉTATION — Variable cible et ROAS J3 :**
> - La distribution **~74% / 26%** est quasi-équilibrée — favorable pour le ML sans SMOTE
> - Le boxplot révèle que le **ROAS J3 est déjà discriminant** : les campagnes qui finissent sous-performantes ont un ROAS J3 médian nettement plus bas que les campagnes saines — ce signal précoce est exactement ce que le modèle NB5 exploitera
> - Les quartiles se chevauchent partiellement — il n'y a pas de seuil magique à J3, c'est la combinaison de plusieurs features qui donnera la puissance prédictive
>
> **MÉTIER :** La séparation des distributions ROAS J3 entre les deux classes confirme l'intuition opérationnelle : les campagnes qui "démarrent mal" finissent généralement mal. C'est l'hypothèse centrale du système d'alerte — elle est validée empiriquement par ce graphique.

---
## Étape 8 — Sélection des features et export

### MÉTHODE — Features retenues vs exclues
On exporte uniquement les colonnes utiles pour le NB3 (SQL Analytics) et le NB5 (ML). Les colonnes exclues :
- **Leakage** : `roas_total`, `ctr_total`, `conversions_total` → disponibles uniquement en fin de campagne
- **Redondantes** : colonnes texte quand l'encodage numérique est disponible
- **Identifiants internes** : `audience_id` (remplacé par ses attributs)

> **Note :** Pour le NB3 (SQL Analytics), certaines colonnes de performance totale sont nécessaires. On les conserve dans l'export mais en les documentant clairement comme 'non utilisables comme features ML'.

In [ ]:
COLS_EXPORT = [
    # Identifiants et métadonnées
    'campagne_id', 'client_id', 'canal', 'objectif_campagne', 'statut',
    'date_debut', 'date_fin', 'pays_cible', 'secteur',
    # Features temporelles campagne
    'duree_jours', 'mois_lancement', 'annee_lancement',
    'mois_sin', 'mois_cos',
    'jour_semaine_lancement', 'est_lancement_weekend',
    # Budget
    'budget_alloue_fcfa', 'ratio_depense_budget',
    'ratio_budget_mensuel', 'depassement_budget',
    # Features signaux précoces J1-J3 (utilisables comme features ML)
    'ctr_j1', 'roas_j1',
    'ctr_j3', 'roas_j3', 'taux_conv_j3', 'cpa_j3',
    'variation_ctr_j1_j3', 'variation_roas_j1_j3',
    'ratio_dep_j3_vs_budget', 'nb_jours_j3',
    'impressions_j3', 'clics_j3', 'conversions_j3',
    # Contexte historique (features ML)
    'taux_sous_perf_canal', 'taux_sous_perf_client', 'taux_sous_perf_objectif',
    # Audience (features ML)
    'taille_estimee', 'type_audience', 'tranche_age', 'genre',
    'indice_saturation_j3',
    # Encodages (features ML)
    'canal_num', 'objectif_num', 'type_audience_num',
    # Performance totale (NB3 uniquement — leakage pour ML)
    'impressions_total', 'clics_total', 'conversions_total',
    'ctr_total', 'roas_total', 'cpa_total', 'taux_conv_total',
    'cpc_total', 'nb_jours_actif',
    # Variable cible
    'campagne_sous_performe',
]

# Filtrer les colonnes disponibles
COLS_EXPORT = [c for c in COLS_EXPORT if c in df_analytics.columns]

df_export = df_analytics[COLS_EXPORT].copy()
df_export.to_csv(f'{SAVE_PATH}marketpulse_analytics.csv', index=False)

print(f'✅ marketpulse_analytics.csv exporté')
print(f'   Dimensions          : {df_export.shape}')
print(f'   Granularité         : 1 ligne = 1 campagne')
print(f'   Nulls résiduels     : {df_export.isnull().sum().sum()}')
print(f'   Localisation        : {SAVE_PATH}')
print(f'\nRépartition variable cible :')
print(df_export['campagne_sous_performe'].value_counts())

---
## Bilan du Notebook 2

### Corrections appliquées

| Anomalie | Nb | Méthode |
|---|---|---|
| Clics > impressions | **4** | impressions = max(impressions, clics) |
| Revenus négatifs → NULL | **3** | Valeur inconnue > valeur fausse |
| ROAS aberrant → recalcul | **2** | revenu_genere / depenses_fcfa |
| CTR/ROAS/CPA recalculés | **tous** | Recalcul universel depuis sources primitives |
| Budget dépassé | **3** | Flag `depassement_budget = 1` (pas de suppression) |
| Valeur achat NULL | **~5** | Médiane par canal_source |

### Features créées

| Feature | Catégorie | Rôle ML |
|---|---|---|
| `ctr_j3`, `roas_j3`, `taux_conv_j3` | Signal précoce J3 | Prédicteurs directs à J3 |
| `variation_ctr_j1_j3`, `variation_roas_j1_j3` | Tendance | Saturation précoce détectée |
| `ratio_dep_j3_vs_budget` | Budget | Consommation précoce anormale |
| `taux_sous_perf_canal/client/objectif` | Historique | Performance passée prédictive |
| `duree_jours`, `est_lancement_weekend` | Temporel | Contexte de lancement |
| `indice_saturation_j3` | Audience | Pression sur le segment |
| `canal_num`, `objectif_num`, `type_audience_num` | Encodage | Variables catégorielles → ML |

**Features leakage documentées et exclues du ML (NB5) :**  
`roas_total`, `ctr_total`, `conversions_total`, `taux_conv_total` — calculées sur l'ensemble de la campagne, inconnues à J3

**Fichier exporté :** `marketpulse_analytics.csv` — 354 lignes × ~45 colonnes — dans `SAVE_PATH`

**Pour le NB3 :** charger `marketpulse_analytics.csv` + les tables de référence dans DuckDB pour les analyses SQL avancées — RANK() canaux, LAG() mensuel, NTILE() campagnes, heatmap performance jour × canal.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.